In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/oct-wave-3-0-credit-card-fraud-detection-challenge/sample_submission.csv
/kaggle/input/competitions/oct-wave-3-0-credit-card-fraud-detection-challenge/train.csv
/kaggle/input/competitions/oct-wave-3-0-credit-card-fraud-detection-challenge/test.csv


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import lightgbm as lgb

# Paths
TRAIN_PATH = '/kaggle/input/competitions/oct-wave-3-0-credit-card-fraud-detection-challenge/train.csv'
TEST_PATH = '/kaggle/input/competitions/oct-wave-3-0-credit-card-fraud-detection-challenge/test.csv'
OUT_PATH = 'submission.csv'

In [3]:
TARGET_COL = 'is_fraud'
ID_COL = 'transaction_id'
RANDOM_STATE = 42

# 1. Load Data
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

# 2. Feature Preparation Function
def prep_features(df, ref_columns=None):
    out = pd.get_dummies(df, columns=['merchant_category'])
    if ref_columns is not None:
        out = out.reindex(columns=ref_columns, fill_value=0)
    return out

In [4]:
y_train = train[TARGET_COL].values
X_train_df = prep_features(train.drop(columns=[TARGET_COL, ID_COL]))
feature_names = X_train_df.columns.tolist()
X_train = X_train_df.values.astype(float)

# Process Test Set
X_test_df = prep_features(test.drop(columns=[ID_COL]), ref_columns=feature_names)
X_test = X_test_df.values.astype(float)
test_ids = test[ID_COL].values

In [5]:
# 3. Class Imbalance Ratio
n_pos = y_train.sum()
scale_pos_weight = (len(y_train) - n_pos) / n_pos

In [6]:
# 4. Instantiate & Train Best LightGBM Model
lgbm_model = lgb.LGBMClassifier(
    n_estimators=150,
    learning_rate=0.1,
    num_leaves=31,
    min_child_samples=5,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    verbosity=-1
)

lgbm_model.fit(X_train, y_train)

LGBMClassifier(min_child_samples=5, n_estimators=150, random_state=42,
               scale_pos_weight=np.float64(65.11570247933884), verbosity=-1)

In [7]:
# 5. Predict on Test Data using Optimal Threshold
FINAL_THRESHOLD = 0.10
test_probs = lgbm_model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= FINAL_THRESHOLD).astype(int)

In [8]:
# 6. Generate Kaggle Submission File
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET_COL: test_preds
})

In [9]:
submission.to_csv(OUT_PATH, index=False)

print(f"Submission successfully saved to: {OUT_PATH}")
print(f"Total predicted frauds in test set: {test_preds.sum()} / {len(test_preds)} ({test_preds.mean():.2%})")

Submission successfully saved to: submission.csv
Total predicted frauds in test set: 30 / 2000 (1.50%)
